# 诗歌生成

# 数据处理

In [2]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('./poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [3]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
       # self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, batch_input_shape=[None, None])
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    # @tf.function
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        inp_emb = self.embed_layer(inp_ids)
        rnn_out = self.rnn_layer(inp_emb)
        logits = self.dense(rnn_out)
        return logits
    
    @tf.function
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
    
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.rnncell.call(inp_emb, state) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [4]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [5]:
# @tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

# @tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y, seqlen)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [6]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.821865
epoch 1 : loss 6.574678
epoch 2 : loss 6.0473595
epoch 3 : loss 5.8840804
epoch 4 : loss 5.749594
epoch 5 : loss 5.572081
epoch 6 : loss 5.527182
epoch 7 : loss 5.385561
epoch 8 : loss 5.3547487
epoch 9 : loss 5.374475


# 生成过程

In [34]:
def gen_sentence():
    state = [tf.random.normal(shape=(1, 128), stddev=0.5), tf.random.normal(shape=(1, 128), stddev=0.5)]
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state)
        collect.append(cur_token.numpy()[0])
    return [id2word[t] for t in collect]
print(''.join(gen_sentence()))

石元缒錏掩讵瓣鱙澄迫琏纣抬圣稗忡樵釭旧菉辟鞍侮如魏竞驴继腊捃针玩分焙业如魏圣生矫湜緉苗苗接蔀莑扉罄塔


# 简单的模型保存和加载功能

In [ ]:
import pickle
import os

# 保存所有训练结果
def save_all(model, optimizer, epoch, loss, word2id, id2word, save_path='poem_model'):
    """
    保存模型、优化器状态和训练信息
    
    Args:
        model: 训练好的模型
        optimizer: 优化器
        epoch: 当前epoch数
        loss: 当前损失值
        word2id: 词到id的映射
        id2word: id到词的映射
        save_path: 保存路径前缀
    """
    # 保存模型权重
    model.save_weights(f'{save_path}.weights.h5')
    
    # 保存训练状态
    state = {
        'epoch': epoch,
        'loss': loss,
        'word2id': word2id,
        'id2word': id2word,
        'vocab_size': len(word2id)
    }
    
    # 保存优化器状态
    if hasattr(optimizer, 'get_weights'):
        state['optimizer_weights'] = optimizer.get_weights()
    
    # 保存状态
    with open(f'{save_path}_state.pkl', 'wb') as f:
        pickle.dump(state, f)
    
    print(f"模型已保存到: {save_path}.weights.h5")
    print(f"训练状态已保存到: {save_path}_state.pkl")


# 加载训练结果
def load_all(model, optimizer, load_path='poem_model'):
    """
    加载模型和训练状态
    
    Args:
        model: 模型实例（需要先创建）
        optimizer: 优化器实例（需要先创建）
        load_path: 加载路径前缀
    
    Returns:
        start_epoch: 开始的epoch数（上次训练完成的epoch+1）
        word2id: 词到id的映射
        id2word: id到词的映射
    """
    # 检查文件是否存在
    weights_file = f'{load_path}_weights.h5'
    state_file = f'{load_path}_state.pkl'
    
    if not os.path.exists(weights_file):
        print(f"未找到模型文件: {weights_file}，将从头开始训练")
        return 0, None, None
    
    if not os.path.exists(state_file):
        print(f"未找到状态文件: {state_file}，将从头开始训练")
        return 0, None, None
    
    # 加载模型权重
    model.load_weights(weights_file)
    print(f"模型权重已加载: {weights_file}")
    
    # 加载训练状态
    with open(state_file, 'rb') as f:
        state = pickle.load(f)
    
    print(f"训练状态已加载:")
    print(f"  - 上次Epoch: {state['epoch']}")
    print(f"  - 上次Loss: {state['loss']:.4f}")
    
    # 加载优化器状态
    if 'optimizer_weights' in state and hasattr(optimizer, 'set_weights'):
        optimizer.set_weights(state['optimizer_weights'])
        print("  - 优化器状态已恢复")
    
    # 返回下次开始的epoch和映射
    start_epoch = state['epoch'] + 1
    word2id = state['word2id']
    id2word = state['id2word']
    
    return start_epoch, word2id, id2word


# 改进的训练函数（支持自动保存和恢复）
def train_with_save(epochs, model, optimizer, ds, word2id, id2word, 
                   save_path='poem_model', save_every=5):
    """
    支持自动保存的训练函数
    
    Args:
        epochs: 总训练轮数
        model: 模型
        optimizer: 优化器
        ds: 数据集
        word2id: 词到id的映射
        id2word: id到词的映射
        save_path: 保存路径
        save_every: 每隔多少个epoch保存一次
    
    Returns:
        model: 训练后的模型
    """
    # 尝试加载已有模型
    start_epoch, loaded_word2id, loaded_id2word = load_all(model, optimizer, save_path)
    
    # 如果加载了之前的词汇表，使用它
    if loaded_word2id is not None:
        word2id = loaded_word2id
        id2word = loaded_id2word
    
    # 开始训练
    for epoch in range(start_epoch, epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch}/{epochs-1}")
        print(f"{'='*50}")
        
        total_loss = 0
        batch_count = 0
        
        for step, (x, y, seqlen) in enumerate(ds):
            loss = train_one_step(model, optimizer, x, y, seqlen)
            total_loss += loss.numpy()
            batch_count += 1
            
            if step % 500 == 0 and step > 0:
                avg_loss = total_loss / batch_count
                print(f"  Step {step}, Loss: {loss.numpy():.4f}, Avg: {avg_loss:.4f}")
        
        avg_loss = total_loss / batch_count
        print(f"Epoch {epoch} 完成, 平均损失: {avg_loss:.4f}")
        
        # 定期保存
        if (epoch + 1) % save_every == 0 or epoch == epochs - 1:
            save_all(model, optimizer, epoch, avg_loss, word2id, id2word, save_path)
            print(f"✓ 检查点已保存 (Epoch {epoch})")
    
    print(f"\n训练完成！共训练 {epochs} 个epoch")
    return model


# 保存当前训练结果
def save_current_model(model, optimizer, epoch, loss, word2id, id2word):
    """快速保存当前模型"""
    save_all(model, optimizer, epoch, loss, word2id, id2word, './poem_model_current')


# 加载之前保存的模型
def load_saved_model(model, optimizer, load_path='poem_model'):
    """快速加载模型"""
    start_epoch, word2id, id2word = load_all(model, optimizer, load_path)
    return start_epoch, word2id, id2word

# 训练后保存
save_current_model(model, optimizer, 9, loss.numpy(), word2id, id2word)

模型已保存到: ./poem_model_current.weights.h5
训练状态已保存到: ./poem_model_current_state.pkl


# 加载模型训练结果

In [26]:
import os
import pickle
import tensorflow as tf

# ==========================================
# 步骤 1：准备一个空壳模型（非常重要）
# ==========================================
# 提示：在加载权重之前，必须先实例化模型和优化器（与你训练时的参数保持一致）
# 如果你已经在当前 Notebook 实例化过了，这步可以跳过。
# model = YourModelClass(...) 
# optimizer = tf.keras.optimizers.Adam(...)

# ⚠️ 强烈建议：做一次前向传播空跑，强制 Keras 构建所有的网络层和权重矩阵
# 否则直接 load_weights 可能会报 "未构建层" 的错误
dummy_input = tf.zeros((1, 1), dtype=tf.int32)
_ = model(dummy_input)


# ==========================================
# 步骤 2：修复并定义正确的加载函数
# ==========================================
def load_all_fixed(model, optimizer, load_path='poem_model'):
    """修复了原代码中保存和读取文件名不一致的 Bug"""
    # 这里的后缀必须和 save_all 里的完全一致
    weights_file = f'{load_path}.weights.h5'  
    state_file = f'{load_path}_state.pkl'
    
    if not os.path.exists(weights_file):
        print(f"未找到模型权重文件: {weights_file}")
        return 0, None, None
    
    if not os.path.exists(state_file):
        print(f"未找到训练状态文件: {state_file}")
        return 0, None, None
    
    # 1. 加载模型权重
    model.load_weights(weights_file)
    print(f"✅ 模型权重已成功加载: {weights_file}")
    
    # 2. 加载训练状态（字典等）
    with open(state_file, 'rb') as f:
        state = pickle.load(f)
    
    print(f"✅ 训练状态已成功加载:")
    print(f"  - 上次训练到的 Epoch: {state['epoch']}")
    print(f"  - 上次 Loss: {state['loss']:.4f}")
    
    # 3. 加载优化器状态（如果你只是想生成诗歌，不需要优化器状态，这里即使失败也不影响）
    if 'optimizer_weights' in state and optimizer is not None and hasattr(optimizer, 'set_weights'):
        try:
            # 优化器必须先进行过一次梯度更新才能 set_weights，如果报错可忽略
            optimizer.set_weights(state['optimizer_weights'])
            print("  - 优化器状态已恢复")
        except Exception as e:
            print("  - 优化器状态跳过恢复 (不影响生成测试)")
    
    return state['epoch'] + 1, state['word2id'], state['id2word']


# ==========================================
# 步骤 3：执行加载并覆盖当前环境的变量
# ==========================================
load_path = './poem_model_current'

start_epoch, loaded_word2id, loaded_id2word = load_all_fixed(
    model=model, 
    optimizer=optimizer, 
    load_path=load_path
)

# 确保加载成功后，覆盖当前的字典！这是防止生成乱码（生僻字）的关键！
if loaded_word2id is not None and loaded_id2word is not None:
    word2id = loaded_word2id
    id2word = loaded_id2word
    print(f"✅ 词典已同步！当前词汇表大小: {len(word2id)}")
else:
    print("❌ 加载失败，请检查文件路径！")

✅ 模型权重已成功加载: ./poem_model_current.weights.h5
✅ 训练状态已成功加载:
  - 上次训练到的 Epoch: 9
  - 上次 Loss: 5.2366
✅ 词典已同步！当前词汇表大小: 6772


# 诗歌生成：带多种生成策略的增强版本

In [30]:
# 带多种生成策略的增强版本
def generate_poem_with_strategies(model, word2id, id2word, start_words, max_length=50):
    """
    使用多种策略从指定开头词汇生成诗歌
    """
    results = {}
    
    # 策略1：贪婪解码（确定性）
    def greedy_generation(start_word):
        state = tf.zeros(shape=(1, 128))
        
        bos_token_name = 'bos' 
        if bos_token_name in word2id:
            bos_token = tf.constant([word2id[bos_token_name]], dtype=tf.int32)
            inp_emb = model.embed_layer(bos_token)
            _, new_states = model.rnncell(inp_emb, states=[state])
            state = new_states[0] if isinstance(new_states, (list, tuple)) else new_states
        
        
        cur_token = tf.constant([word2id[start_word]], dtype=tf.int32)
        collect = [start_word]

        for _ in range(max_length - 1):
            inp_emb = model.embed_layer(cur_token)
            # 【修改点2】：同上
            h, new_states = model.rnncell(inp_emb, states=[state])
            state = new_states[0] if isinstance(new_states, (list, tuple)) else new_states
            
            logits = model.dense(h)
            next_token_idx = tf.argmax(logits, axis=-1).numpy()[0]
            next_word = id2word[next_token_idx]
            
            if next_word == 'eos':
                break
            
            collect.append(next_word)
            cur_token = tf.constant([next_token_idx], dtype=tf.int32)
        
        return ''.join(collect)
    
    # 策略2：带温度的随机采样（temperature=0.8，更保守）
    def temperature_sampling(start_word, temp=0.8):
        state = tf.zeros(shape=(1, 128))

        bos_token_name = 'bos' 
        if bos_token_name in word2id:
            bos_token = tf.constant([word2id[bos_token_name]], dtype=tf.int32)
            inp_emb = model.embed_layer(bos_token)
            _, new_states = model.rnncell(inp_emb, states=[state])
            state = new_states[0] if isinstance(new_states, (list, tuple)) else new_states

        cur_token = tf.constant([word2id[start_word]], dtype=tf.int32)
        collect = [start_word]
        
        for _ in range(max_length - 1):
            inp_emb = model.embed_layer(cur_token)
            # 【修改点4】
            h, new_states = model.rnncell(inp_emb, states=[state])
            state = new_states[0] if isinstance(new_states, (list, tuple)) else new_states
            
            logits = model.dense(h) / temp
            
            probs = tf.nn.softmax(logits, axis=-1)
            next_token_idx = tf.random.categorical(tf.math.log(probs + 1e-10), num_samples=1)
            next_token_idx = next_token_idx[0, 0].numpy()
            next_word = id2word[next_token_idx]
            
            if next_word == 'eos':
                break
            
            collect.append(next_word)
            cur_token = tf.constant([next_token_idx], dtype=tf.int32)
        
        return ''.join(collect)
    
    # 策略3：Top-k采样（只从概率最高的k个词中采样）
    def topk_sampling(start_word, k=5):
        state = tf.zeros(shape=(1, 128))
        
        bos_token_name = 'bos' 
        if bos_token_name in word2id:
            bos_token = tf.constant([word2id[bos_token_name]], dtype=tf.int32)
            inp_emb = model.embed_layer(bos_token)
            _, new_states = model.rnncell(inp_emb, states=[state])
            state = new_states[0] if isinstance(new_states, (list, tuple)) else new_states
        
        cur_token = tf.constant([word2id[start_word]], dtype=tf.int32)
        collect = [start_word]
        
        for _ in range(max_length - 1):
            inp_emb = model.embed_layer(cur_token)
            # 【修改点6】
            h, new_states = model.rnncell(inp_emb, states=[state])
            state = new_states[0] if isinstance(new_states, (list, tuple)) else new_states
            
            logits = model.dense(h)  # [1, vocab_size]
            
            # Top-k采样
            logits = tf.squeeze(logits, axis=0)  # [vocab_size]
            top_k_values, top_k_indices = tf.math.top_k(logits, k=k)
            
            # 重新归一化top-k的概率
            top_k_probs = tf.nn.softmax(top_k_values)
            
            # 从top-k中采样
            sampled_idx = tf.random.categorical(tf.math.log([top_k_probs]), num_samples=1)
            sampled_idx = sampled_idx[0, 0].numpy()
            next_token_idx = top_k_indices[sampled_idx].numpy()
            
            next_word = id2word[next_token_idx]
            
            if next_word == 'eos':
                break
            
            collect.append(next_word)
            cur_token = tf.constant([next_token_idx], dtype=tf.int32)
        
        return ''.join(collect)
    
    # ---------------- 运行生成流程 ----------------
    results['greedy'] = {}
    results['temperature_0.8'] = {}
    results['topk_5'] = {}
    
    # 【小贴士】：为了确保模型权重已经被加载/构建，在手动拆解模型调用前，最好先做一次 dummy forward pass
    dummy_input = tf.zeros((1, 1), dtype=tf.int32)
    _ = model(dummy_input)  # 这行代码可以强制整个模型构建完毕，避免遗漏
    
    for start_word in start_words:
        if start_word not in word2id:
            print(f"警告: '{start_word}' 不在词汇表中，跳过")
            continue
        
        print(f"正在为开头词 '{start_word}' 生成诗歌...")
        results['greedy'][start_word] = greedy_generation(start_word)
        results['temperature_0.8'][start_word] = temperature_sampling(start_word, temp=0.8)
        results['topk_5'][start_word] = topk_sampling(start_word, k=5)
    
    return results

# 打印生成结果的函数
def print_poems(results, start_words):
    """优雅地打印生成的诗歌"""
    print("=" * 80)
    print("诗歌生成结果")
    print("=" * 80)
    
    for strategy_name, poems in results.items():
        print(f"\n【生成策略：{strategy_name}】")
        print("-" * 60)
        for start_word in start_words:
            if start_word in poems:
                poem = poems[start_word]
                print(f"以 '{start_word}' 开头的诗歌: {poem}")
                # 如果诗歌太长，适当换行显示
                if len(poem) > 30:
                    # 按每15个字换行显示
                    for i in range(0, len(poem), 15):
                        print(f"  {poem[i:i+15]}")
                print()
        print("-" * 60)


# 执行生成
print("开始生成诗歌...")

# 定义要生成的开头词汇
start_words = ['日', '红', '山', '夜', '湖', '海', '月']

# 使用多种策略生成诗歌
generated_results = generate_poem_with_strategies(model, word2id, id2word, start_words, max_length=50)

# 打印结果
print_poems(generated_results, start_words)

开始生成诗歌...
正在为开头词 '日' 生成诗歌...
正在为开头词 '红' 生成诗歌...
正在为开头词 '山' 生成诗歌...
正在为开头词 '夜' 生成诗歌...
正在为开头词 '湖' 生成诗歌...
正在为开头词 '海' 生成诗歌...
正在为开头词 '月' 生成诗歌...
诗歌生成结果

【生成策略：greedy】
------------------------------------------------------------
以 '日' 开头的诗歌: 日日无人去，春风落月中。

以 '红' 开头的诗歌: 红叶满山花，一枝不可知。

以 '山' 开头的诗歌: 山上春风起，春风落月中。

以 '夜' 开头的诗歌: 夜来春雨满，春雨满山边。

以 '湖' 开头的诗歌: 湖上春风起，春风满树花。

以 '海' 开头的诗歌: 海上春风起，春风落月中。

以 '月' 开头的诗歌: 月上春风吹，风风满树花。

------------------------------------------------------------

【生成策略：temperature_0.8】
------------------------------------------------------------
以 '日' 开头的诗歌: 日更烽管锦，烧风碧青岑。三夜犹自歇，吟风须自闻。朱屏多伯知，雅要扫金闱。
  日更烽管锦，烧风碧青岑。三夜犹
  自歇，吟风须自闻。朱屏多伯知，
  雅要扫金闱。

以 '红' 开头的诗歌: 红雷谩横燕，千里十山前。向晚临泉月，山飞尚路空。

以 '山' 开头的诗歌: 山里尘，马间时。莫得孤空绪，心间自有年。明朝不可厌，闲径不相游。
  山里尘，马间时。莫得孤空绪，心
  间自有年。明朝不可厌，闲径不相
  游。

以 '夜' 开头的诗歌: 夜落时来塞，光明日发尘。左知群不见，寒水不同肠。

以 '湖' 开头的诗歌: 湖水未知年，夜深七月云。何情留岁月，突彩犯垂厘。

以 '海' 开头的诗歌: 海上今来隔雪中，几年无夜望无人。

以 '月' 开头的诗歌: 月中骑翠漫，时之一个稀。__翻作右焉章，天宗扰变》）

------------------------------------------------------------

【生成策